# Acute Leukemia (ALL / AML) Classification from Gene Expression Data
### Neural Networks and Fuzzy Logic Course Project (ANN601) — Syrian Virtual University (F25)

**Project Idea:** A Multi-Layer Perceptron (`MLP`) built with `PyTorch` to distinguish `ALL` from `AML`, benchmarked against classical machine learning models under a strict evaluation protocol designed to prevent data leakage given the small sample size (72 samples).

## 1) Setup

In [ ]:
import os, json, random, shutil, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_validate, cross_val_predict, train_test_split)
from sklearn.metrics import (accuracy_score, roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from scipy.stats import ttest_ind

warnings.filterwarnings("ignore")
SEED = 42

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

set_seed()
os.makedirs("results", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})
print("PyTorch:", torch.__version__)

## 2) Load Dataset

**Recommended option:** Search Kaggle for *Gene expression dataset (Golub et al.)* and upload the three files:
`data_set_ALL_AML_train.csv`, `data_set_ALL_AML_independent.csv`, and `actual.csv`.

Run the next cell **only if the files are not already present** (it opens a file upload dialog in Colab). If you have a single file named `golub.csv` (columns = genes + a `label` column), upload that instead.

In [ ]:
# (Optional) Upload files in Colab
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
except ImportError:
    print("Not running in Colab: place the dataset files next to the notebook manually.")

In [ ]:
def _read_kaggle_expr(path):
    """Read a Kaggle-formatted Golub file (rows = genes, columns = patients + call columns)."""
    df = pd.read_csv(path)
    df = df.drop(columns=[c for c in df.columns if str(c).startswith("call")])
    acc = df["Gene Accession Number"].astype(str)
    cum = acc.groupby(acc).cumcount()
    acc = acc.where(cum == 0, acc + "_dup" + cum.astype(str))      # Make accession IDs unique
    desc = pd.Series(df["Gene Description"].values, index=acc.values)
    expr = df.drop(columns=["Gene Description", "Gene Accession Number"])
    expr.index = acc.values
    expr.columns = expr.columns.astype(int)
    return expr, desc


def load_golub():
    f_train, f_test, f_lab = ("data_set_ALL_AML_train.csv",
                              "data_set_ALL_AML_independent.csv", "actual.csv")
    # Option 1: Kaggle files
    if all(os.path.exists(f) for f in (f_train, f_test, f_lab)):
        tr, desc = _read_kaggle_expr(f_train)
        te, _ = _read_kaggle_expr(f_test)
        expr = pd.concat([tr, te], axis=1).sort_index(axis=1)
        lab = pd.read_csv(f_lab).set_index("patient")["cancer"]
        X = expr.T                                   # rows = samples, columns = genes
        y = lab.loc[X.index].astype(str).str.strip().values
        orig_train = np.asarray(X.index.isin(list(tr.columns)))
        return X, y, desc, orig_train, "Kaggle (train + independent)"
    # Option 2: Single combined CSV file
    if os.path.exists("golub.csv"):
        df = pd.read_csv("golub.csv")
        col = "label" if "label" in df.columns else df.columns[-1]
        y = df[col].astype(str).values
        X = df.drop(columns=[col]).select_dtypes("number")
        return X, y, None, None, "golub.csv"
    # Option 3: OpenML fallback (name/version may vary; try if local files are unavailable)
    try:
        from sklearn.datasets import fetch_openml
        d = fetch_openml(name="leukemia", version=1, as_frame=True, parser="auto")
        X = d.data.select_dtypes("number")
        return X, d.target.astype(str).values, None, None, "OpenML"
    except Exception as e:
        raise RuntimeError(
            "Failed to load dataset. Upload the three Kaggle files (see previous cell).\n" + str(e))


X, y_raw, gene_desc, orig_train, SOURCE = load_golub()
le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = list(le.classes_)

print("Source:", SOURCE)
print("Shape (samples x genes):", X.shape)
print("Classes:", dict(zip(class_names, np.bincount(y).tolist())))
n_missing = int(X.isna().sum().sum())
print("Missing values:", n_missing)
if n_missing:
    X = X.fillna(X.median())
assert len(class_names) == 2, "This project is designed for binary classification (ALL/AML)"

## 3) Preprocessing (Without Learning from Data)
Clip intensity values (`floor=20`, `ceiling=16000`) followed by a `log2` transform as in the original Golub study. This step is **deterministic** and does not learn from class labels, so it causes no data leakage. Feature selection and scaling will instead happen **inside** each cross-validation fold.

In [ ]:
print("Range before preprocessing: [%.1f , %.1f]" % (X.values.min(), X.values.max()))
if X.values.max() > 100:                       # Raw microarray intensity values
    X = np.log2(X.clip(lower=20, upper=16000))
    print("Applied clip + log2")
else:
    print("Data appears pre-processed; log2 was not applied")
print("Range after preprocessing:  [%.2f , %.2f]" % (X.values.min(), X.values.max()))

n_after_var = int(VarianceThreshold(0.1).fit(X).get_support().sum())
print("Genes remaining after low-variance filter (diagnostic on full data only):", n_after_var)
assert n_after_var > 500, "Lower max k in experiments or reduce the variance threshold"

## 4) Exploratory Data Analysis (EDA)
These plots are **descriptive only** (not used for model selection), so they do not introduce leakage.

In [ ]:
Z = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=SEED).fit(Z)
P = pca.transform(Z)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
counts = np.bincount(y)
ax[0].bar(class_names, counts, color=["#2a7f8e", "#c8553d"])
for i, c in enumerate(counts):
    ax[0].text(i, c + 0.5, str(c), ha="center")
ax[0].set_title("Class distribution"); ax[0].set_ylabel("Samples")
for c, col in zip(range(2), ["#2a7f8e", "#c8553d"]):
    ax[1].scatter(P[y == c, 0], P[y == c, 1], c=col, label=class_names[c], alpha=.8)
ax[1].set_xlabel("PC1 (%.1f%%)" % (pca.explained_variance_ratio_[0] * 100))
ax[1].set_ylabel("PC2 (%.1f%%)" % (pca.explained_variance_ratio_[1] * 100))
ax[1].set_title("PCA of samples"); ax[1].legend()
plt.tight_layout(); plt.savefig("results/fig1_eda.png"); plt.show()

## 5) Quick Differential Expression (Welch's t-test + FDR Correction)
Connects the project to bioinformatics: how many genes differ significantly between the two classes? (Descriptive only, not used for model feature selection.)

In [ ]:
def bh_fdr(p):
    p = np.asarray(p); n = len(p); order = np.argsort(p)
    ranked = p[order] * n / np.arange(1, n + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n); out[order] = np.minimum(ranked, 1.0)
    return out

a, b = X.values[y == 0], X.values[y == 1]
t, p = ttest_ind(a, b, equal_var=False)
p = np.nan_to_num(p, nan=1.0)
de = pd.DataFrame({"log2FC": a.mean(0) - b.mean(0), "p": p, "FDR": bh_fdr(p)}, index=X.columns)
if gene_desc is not None:
    de["description"] = gene_desc.reindex(de.index).values
de = de.sort_values("FDR")
n_sig = int(((de.FDR < 0.05) & (de.log2FC.abs() > 1)).sum())
print(f"Significant genes (FDR<0.05 and |log2FC|>1): {n_sig} out of {len(de)}")
de.head(20).to_csv("results/top20_differential_genes.csv")

plt.figure(figsize=(6, 4.5))
sig = (de.FDR < 0.05) & (de.log2FC.abs() > 1)
plt.scatter(de.log2FC[~sig], -np.log10(de.p[~sig] + 1e-300), s=4, c="#999999", alpha=.5)
plt.scatter(de.log2FC[sig], -np.log10(de.p[sig] + 1e-300), s=6, c="#c8553d", alpha=.8)
plt.xlabel(f"log2 fold-change ({class_names[0]} - {class_names[1]})"); plt.ylabel("-log10 p-value")
plt.title("Volcano plot"); plt.tight_layout(); plt.savefig("results/fig2_volcano.png"); plt.show()
de.head(10)

## 6) Model: PyTorch MLP (Scikit-Learn Compatible)
We wrap the neural network in a scikit-learn-compatible estimator so it can be placed inside a `Pipeline` and evaluated via cross-validation without data leakage.

Default architecture: `Linear -> BatchNorm -> ReLU -> Dropout` twice (`128` then `64` hidden units), followed by a 2-unit output layer. Optimizer: `Adam` with weight decay; loss function: `CrossEntropyLoss`.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in, hidden=(128, 64), dropout=0.4, n_classes=2):
        super().__init__()
        layers, prev = [], d_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class TorchMLPClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, hidden=(128, 64), dropout=0.4, lr=1e-3, weight_decay=1e-3,
                 epochs=100, batch_size=16, random_state=SEED):
        self.hidden = hidden
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state

    @staticmethod
    def _tensor(X):
        return torch.as_tensor(np.asarray(X, dtype=np.float32))

    def fit(self, X, y, eval_set=None):
        set_seed(self.random_state)
        Xt = self._tensor(X)
        self.classes_ = np.unique(y)
        yt = torch.as_tensor(np.searchsorted(self.classes_, y), dtype=torch.long)
        self.model_ = MLP(Xt.shape[1], self.hidden, self.dropout, len(self.classes_))
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.CrossEntropyLoss()
        self.history_ = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        if eval_set is not None:
            Xv = self._tensor(eval_set[0])
            yv = torch.as_tensor(np.searchsorted(self.classes_, eval_set[1]), dtype=torch.long)
        n = len(Xt)
        for _ in range(self.epochs):
            self.model_.train()
            perm = torch.randperm(n)
            total = 0.0
            for i in range(0, n, self.batch_size):
                idx = perm[i:i + self.batch_size]
                if len(idx) < 2:                  # BatchNorm requires at least 2 samples
                    continue
                opt.zero_grad()
                loss = loss_fn(self.model_(Xt[idx]), yt[idx])
                loss.backward()
                opt.step()
                total += loss.item() * len(idx)
            self.history_["train_loss"].append(total / n)
            if eval_set is not None:
                self.model_.eval()
                with torch.no_grad():
                    out_tr, out_v = self.model_(Xt), self.model_(Xv)
                    self.history_["train_acc"].append((out_tr.argmax(1) == yt).float().mean().item())
                    self.history_["val_acc"].append((out_v.argmax(1) == yv).float().mean().item())
                    self.history_["val_loss"].append(loss_fn(out_v, yv).item())
        return self

    def predict_proba(self, X):
        self.model_.eval()
        with torch.no_grad():
            return torch.softmax(self.model_(self._tensor(X)), dim=1).numpy()

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

## 7) Modeling Pipeline and Baseline Models
Every step learns from **training data only** inside each fold: low-variance gene removal -> top-`k` gene selection (ANOVA F-test) -> standardization -> classifier.

In [ ]:
K_DEFAULT = 100

def make_pipe(clf, k=K_DEFAULT):
    return Pipeline([
        ("var", VarianceThreshold(0.1)),
        ("select", SelectKBest(f_classif, k=k)),
        ("scale", StandardScaler()),
        ("clf", clf),
    ])

def get_models(k=K_DEFAULT):
    return {
        "Logistic Regression": make_pipe(LogisticRegression(C=0.1, max_iter=5000), k),
        "SVM (linear)":        make_pipe(SVC(kernel="linear", C=0.1, probability=True, random_state=SEED), k),
        "Random Forest":       make_pipe(RandomForestClassifier(n_estimators=300, random_state=SEED), k),
        "MLP (PyTorch)":       make_pipe(TorchMLPClassifier(), k),
    }

# 5-fold x 5-repeat stratified CV = 25 evaluations (more reliable than a single split on 72 samples)
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
SCORING = ["accuracy", "balanced_accuracy", "f1_macro", "roc_auc"]

def evaluate(name, pipe):
    r = cross_validate(pipe, X, y, cv=CV, scoring=SCORING, return_train_score=True, n_jobs=1)
    row = {"model": name}
    for m in SCORING:
        row[m + "_mean"] = r["test_" + m].mean()
        row[m + "_std"] = r["test_" + m].std()
    row["train_accuracy_mean"] = r["train_accuracy"].mean()
    return row

def fmt_table(df):
    out = pd.DataFrame({"model": df["model"]})
    for m in SCORING:
        out[m] = [f"{a*100:.1f} ± {s*100:.1f}" for a, s in zip(df[m + "_mean"], df[m + "_std"])]
    out["train_accuracy"] = [f"{a*100:.1f}" for a in df["train_accuracy_mean"]]
    return out

## 8) Main Model Comparison (May Take a Few Minutes)

In [ ]:
rows = []
for name, pipe in get_models().items():
    print("Evaluating:", name, "...")
    rows.append(evaluate(name, pipe))
main_df = pd.DataFrame(rows)
main_df.to_csv("results/model_comparison_raw.csv", index=False)
main_tbl = fmt_table(main_df)
main_tbl.to_csv("results/model_comparison.csv", index=False)
print("\nResults (percentage: mean ± std across 25 folds)")
print(main_tbl.to_string(index=False))

# Plot comparison
metrics = ["accuracy", "f1_macro", "roc_auc"]
w = 0.25
plt.figure(figsize=(8, 4.2))
for i, m in enumerate(metrics):
    plt.bar(np.arange(len(main_df)) + i * w, main_df[m + "_mean"] * 100, w,
            yerr=main_df[m + "_std"] * 100, capsize=3, label=m)
plt.xticks(np.arange(len(main_df)) + w, main_df["model"], rotation=15)
plt.ylim(50, 102); plt.ylabel("%"); plt.title(f"Model comparison (k={K_DEFAULT} genes, 5x5 repeated CV)")
plt.legend(loc="lower right"); plt.tight_layout(); plt.savefig("results/fig3_model_comparison.png"); plt.show()

## 9) ROC Curves and Confusion Matrix (Out-of-Fold Predictions)
Each sample is predicted by a model that did not see it during training.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = {}
plt.figure(figsize=(5.5, 5))
for name, pipe in get_models().items():
    proba = cross_val_predict(pipe, X, y, cv=skf, method="predict_proba")[:, 1]
    oof[name] = proba
    fpr, tpr, _ = roc_curve(y, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y, proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves (out-of-fold)"); plt.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig("results/fig4_roc.png"); plt.show()

pred_mlp = (oof["MLP (PyTorch)"] >= 0.5).astype(int)
cm = confusion_matrix(y, pred_mlp)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues", values_format="d")
plt.title("MLP confusion matrix (out-of-fold)")
plt.savefig("results/fig5_confusion_mlp.png"); plt.show()
print(classification_report(y, pred_mlp, target_names=class_names, digits=3))

## 10) Ablation Studies: Number of Genes and Network Architecture

In [ ]:
# (a) Effect of number of selected genes k on MLP and Logistic Regression
k_values = [10, 20, 50, 100, 250, 500]
rows = []
for k in k_values:
    for name, clf in [("MLP (PyTorch)", TorchMLPClassifier()),
                      ("Logistic Regression", LogisticRegression(C=0.1, max_iter=5000))]:
        r = evaluate(name, make_pipe(clf, k)); r["k"] = k; rows.append(r)
        print(f"k={k:<4} {name:<20} acc={r['accuracy_mean']*100:.1f} ± {r['accuracy_std']*100:.1f}")
abl_k = pd.DataFrame(rows)
abl_k.to_csv("results/ablation_genes_raw.csv", index=False)

plt.figure(figsize=(6, 4))
for name in abl_k["model"].unique():
    d = abl_k[abl_k["model"] == name]
    plt.errorbar(d["k"], d["accuracy_mean"] * 100, yerr=d["accuracy_std"] * 100, marker="o", capsize=3, label=name)
plt.xscale("log"); plt.xlabel("Number of selected genes (k)"); plt.ylabel("CV accuracy (%)")
plt.title("Effect of the number of genes"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/fig6_ablation_genes.png"); plt.show()

In [ ]:
# (b) Effect of network architecture and Dropout (fixed k = 100)
arch = {
    "1 hidden (64), dropout 0.4":          dict(hidden=(64,), dropout=0.4),
    "2 hidden (128,64), dropout 0.4":      dict(hidden=(128, 64), dropout=0.4),
    "2 hidden (128,64), no dropout":       dict(hidden=(128, 64), dropout=0.0),
    "3 hidden (256,128,64), dropout 0.4":  dict(hidden=(256, 128, 64), dropout=0.4),
}
rows = []
for name, kw in arch.items():
    r = evaluate(name, make_pipe(TorchMLPClassifier(**kw), K_DEFAULT)); rows.append(r)
    print(f"{name:<38} acc={r['accuracy_mean']*100:.1f} ± {r['accuracy_std']*100:.1f}  (train acc={r['train_accuracy_mean']*100:.1f})")
abl_arch = pd.DataFrame(rows)
abl_arch.to_csv("results/ablation_architecture_raw.csv", index=False)
fmt_table(abl_arch).to_csv("results/ablation_architecture.csv", index=False)

## 11) Learning Curves (Train and Validation Loss & Accuracy)
We split the data once (75/25) to visualize training dynamics. (Primary reported figures come from repeated cross-validation above, not this single split.)

In [ ]:
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
pipe = make_pipe(TorchMLPClassifier(epochs=150), K_DEFAULT)
pre = Pipeline(pipe.steps[:-1]).fit(Xtr, ytr)            # Fit preprocessing on training split only
clf = pipe.named_steps["clf"].fit(pre.transform(Xtr), ytr, eval_set=(pre.transform(Xva), yva))
h = clf.history_

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h["train_loss"], label="train"); ax[0].plot(h["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(h["train_acc"], label="train"); ax[1].plot(h["val_acc"], label="validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/fig7_learning_curves.png"); plt.show()
print("Final validation accuracy on this split: %.3f (validation samples = %d)" % (h["val_acc"][-1], len(yva)))

## 12) (Optional) Evaluation on Original Golub Train/Test Split
38 training samples and 34 independent test samples. Runs only if Kaggle files were loaded.

In [ ]:
orig_rows = []
if orig_train is not None:
    for name, pipe in get_models().items():
        m = clone(pipe).fit(X[orig_train], y[orig_train])
        proba = m.predict_proba(X[~orig_train])[:, 1]
        acc = accuracy_score(y[~orig_train], (proba >= .5).astype(int))
        auc = roc_auc_score(y[~orig_train], proba)
        orig_rows.append({"model": name, "test_accuracy": round(acc, 3), "test_auc": round(auc, 3)})
    orig_df = pd.DataFrame(orig_rows)
    orig_df.to_csv("results/original_split.csv", index=False)
    print(f"Trained on {int(orig_train.sum())} samples, tested on {int((~orig_train).sum())} independent samples:")
    print(orig_df.to_string(index=False))
else:
    print("Skipped: original split is not available with this data source.")

## 13) Gene Stability (Biomarkers)
How often is each gene selected among the top 20 across the 25 training folds? Frequently selected genes are candidates for biological interpretation in the discussion.

In [ ]:
counter = Counter()
splits = list(CV.split(X, y))
for tr, _ in splits:
    Xt_ = X.iloc[tr]
    vt = VarianceThreshold(0.1).fit(Xt_)
    cols = X.columns[vt.get_support()]
    sel = SelectKBest(f_classif, k=20).fit(Xt_[cols], y[tr])
    counter.update(cols[sel.get_support()])

stab = pd.DataFrame({"gene": list(counter.keys()), "times_selected": list(counter.values())})
stab["frequency_%"] = (stab["times_selected"] / len(splits) * 100).round(1)
if gene_desc is not None:
    stab["description"] = gene_desc.reindex(stab["gene"]).values
stab = stab.sort_values("times_selected", ascending=False).head(20).reset_index(drop=True)
stab.to_csv("results/stable_genes_top20.csv", index=False)
pd.set_option("display.max_colwidth", 70)
print(stab.to_string(index=False))

## 14) Save Summary and Download Results

In [ ]:
summary = {
    "source": SOURCE,
    "n_samples": int(X.shape[0]), "n_genes_raw": int(X.shape[1]),
    "class_counts": {c: int(n) for c, n in zip(class_names, np.bincount(y))},
    "n_significant_DE_genes": n_sig,
    "cv": "RepeatedStratifiedKFold 5x5", "k_default": K_DEFAULT,
    "main_comparison": fmt_table(main_df).to_dict(orient="records"),
    "ablation_architecture": fmt_table(abl_arch).to_dict(orient="records"),
    "original_split": orig_rows,
}
with open("results/summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

shutil.make_archive("results_package", "zip", "results")
print("Files in results/:", sorted(os.listdir("results")))
try:
    from google.colab import files
    files.download("results_package.zip")
except ImportError:
    pass